# Cell 1: Environment Installation

In [ ]:
# Updated Cell 1
import subprocess
import sys

print("Installing fully compatible model training dependencies...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.44.2",  # Sweet spot version: fixes both the token and cache import bugs
    "datasets",
    "evaluate",
    "sacrebleu",
    "accelerate",
    "sentencepiece",
], check=True)

print("✅ Done. Go to the top menu: Runtime -> Restart session, then run Cell 2!")

Installing fully compatible model training dependencies...
✅ Done. Go to the top menu: Runtime -> Restart session, then run Cell 2!


# Cell 2: Imports, Environment Setup, and Google Drive Connection

In [ ]:
import json
import gc
import os
import numpy as np
import torch
from pathlib import Path
from difflib import SequenceMatcher
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from datasets import Dataset
import evaluate
from google.colab import drive

# 1. Mount Google Drive to keep data secure
drive.mount('/content/drive')

# 2. Paths and Specialized Code Architecture Constants
MODEL  = "Salesforce/codet5-base" # Upgraded to native code architecture
PREFIX = "fix java bug: "
MAX_IN  = 256
MAX_OUT = 256

# Output and checkpoints go to Drive so you don't lose progress if disconnected
OUT  = Path("/content/drive/MyDrive/java_bugfix_project/codet5_v1")
OUT.mkdir(parents=True, exist_ok=True)
DATA = Path("/content/drive/MyDrive/java_bugfix_project")

print(f"Transformers: {__import__('transformers').__version__}")
print(f"Accelerate:   {__import__('accelerate').__version__}")
print(f"PyTorch:      {torch.__version__}")
print(f"GPU:          {torch.cuda.get_device_name(0)}")
print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Data files:   {list(DATA.iterdir()) if DATA.exists() else 'PATH NOT FOUND'}")
print("✅ Configuration and Drive Setup Complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Transformers: 4.44.2
Accelerate:   1.14.0
PyTorch:      2.11.0+cu128
GPU:          Tesla T4
VRAM:         15.6 GB
Data files:   [PosixPath('/content/drive/MyDrive/java_bugfix_project/train_diff.jsonl'), PosixPath('/content/drive/MyDrive/java_bugfix_project/val_diff.jsonl'), PosixPath('/content/drive/MyDrive/java_bugfix_project/test_diff.jsonl'), PosixPath('/content/drive/MyDrive/java_bugfix_project/codet5_v1')]
✅ Configuration and Drive Setup Complete.


# Cell 3: Loading Data Pairs

In [ ]:
def load_jsonl(path: Path, max_samples: int = None) -> list:
    pairs = []
    if not path.exists():
        print(f"🚨 Missing file error: Could not find {path}")
        return pairs
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                pairs.append(json.loads(line))
            except Exception:
                continue
            if max_samples and len(pairs) >= max_samples:
                break
    return pairs

train_raw = load_jsonl(DATA / "train_diff.jsonl")
val_raw   = load_jsonl(DATA / "val_diff.jsonl")
test_raw  = load_jsonl(DATA / "test_diff.jsonl")

print(f"Loaded datasets successfully:")
print(f"  Train: {len(train_raw):,} pairs")
print(f"  Val:   {len(val_raw):,} pairs")
print(f"  Test:  {len(test_raw):,} pairs")

Loaded datasets successfully:
  Train: 26,754 pairs
  Val:   6,293 pairs
  Test:  6,079 pairs


# Cell 4: Tokenization Pipeline

In [ ]:
# Updated Cell 4 - Strict Length Trimming for Speed
MAX_IN = 256   # Clamps input code lengths
MAX_OUT = 128  # Clamps output fix lengths

tokenizer = AutoTokenizer.from_pretrained(MODEL)
print(f"✅ Loaded Tokenizer: {tokenizer.__class__.__name__}")

train_ds = Dataset.from_list(train_raw)
val_ds   = Dataset.from_list(val_raw)
test_ds  = Dataset.from_list(test_raw)

def tokenize(batch):
    inputs = tokenizer(
        [PREFIX + str(b) for b in batch["buggy_code"]],
        max_length=MAX_IN,
        truncation=True,  # Crucial: Cuts off text past 256 tokens
        padding=False,    # Let the collator pad dynamically per batch
    )
    targets = tokenizer(
        text_target=[str(f) for f in batch["fixed_code"]],
        max_length=MAX_OUT,
        truncation=True,  # Crucial: Cuts off text past 128 tokens
        padding=False,
    )
    inputs["labels"] = targets["input_ids"]
    return inputs

print("Tokenizing and strictly trimming datasets...")
train_ds = train_ds.map(tokenize, batched=True, batch_size=256, num_proc=2, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(tokenize, batched=True, batch_size=256, num_proc=2, remove_columns=val_ds.column_names)
test_ds  = test_ds.map(tokenize, batched=True, batch_size=256, num_proc=2, remove_columns=test_ds.column_names)

print("\n✅ Tokenization Complete. Data is optimized for speed.")

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Loaded Tokenizer: RobertaTokenizerFast
Tokenizing and strictly trimming datasets...


Map (num_proc=2):   0%|          | 0/26754 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/6293 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/6079 [00:00<?, ? examples/s]


✅ Tokenization Complete. Data is optimized for speed.


# Cell 5: Initialize CodeT5 Model

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

total_params = model.num_parameters()
print(f"✅ CodeT5-Base Loaded:")
print(f"   Total params:     {total_params:,}")
print(f"   Model size:       ~{total_params * 4 / 1e9:.2f} GB (fp32)")

✅ CodeT5-Base Loaded:
   Total params:     222,882,048
   Model size:       ~0.89 GB (fp32)


# Cell 6: Metrics Configuration

In [ ]:
bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    preds = np.clip(preds, 0, tokenizer.vocab_size - 1).astype(np.int32)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.clip(labels, 0, tokenizer.vocab_size - 1).astype(np.int32)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]

    bleu  = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    exact = sum(p == l[0] for p, l in zip(decoded_preds, decoded_labels))
    sim   = sum(
        SequenceMatcher(None, p, l[0]).ratio()
        for p, l in zip(decoded_preds, decoded_labels)
    ) / max(len(decoded_preds), 1)

    return {
        "bleu":            round(bleu["score"], 2),
        "exact_match_pct": round(exact / len(decoded_preds) * 100, 2),
        "similarity_pct":  round(sim * 100, 2),
    }

print("✅ Evaluation metrics registered.")

✅ Evaluation metrics registered.


# Cell 7: Training Arguments Setup

In [ ]:
# Absolute Best Practices Stable Speed Scheme
args = Seq2SeqTrainingArguments(
    output_dir=str(OUT),
    num_train_epochs=3,
    warmup_steps=200,

    # ── TRAINING (STABLE & FAST AT 0.61 IT/S) ──
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}, # Fixes the PyTorch warning

    # ── EVALUATION (COMPLETELY OOM-PROOF) ──
    per_device_eval_batch_size=1,    # Drops evaluation memory overhead to baseline minimum
    prediction_loss_only=True,       # Forces HF to only track loss scalar, zero tensor hoarding
    eval_do_concat_batches=False,

    learning_rate=5e-5,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    fp16=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    predict_with_generate=False,
    save_total_limit=2,
    logging_steps=100,
    report_to="none",
    seed=42,
)

# Cell 8: Execution Loop (Forced Clean Execution)

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("🧹 GPU Memory cleared!")

🧹 GPU Memory cleared!


In [ ]:
# Updated Cell 8 Setup
collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,      # 💡 Changed from 'processing_class' to 'tokenizer'
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("🚀 Launching Clean Model Training Loop...")
train_result = trainer.train()

print(f"\n✅ Training Completed Successfully.")
print(f"   Final Training Loss: {train_result.training_loss:.4f}")
print(f"   Total Computed Steps: {train_result.global_step:,}")

🚀 Launching Clean Model Training Loop...


Step,Training Loss,Validation Loss
500,0.702700,0.740294
1000,0.758500,0.708071
1500,0.727900,0.692799
2000,0.702300,0.693779
2500,0.702000,0.691474


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



✅ Training Completed Successfully.
   Final Training Loss: 0.7458
   Total Computed Steps: 2,508


# Cell 9: Model Preservation Save

In [ ]:
FINAL = OUT / "final_production_model"
FINAL.mkdir(exist_ok=True)

model.save_pretrained(str(FINAL))
tokenizer.save_pretrained(str(FINAL))

print(f"✅ Model weights saved securely in Google Drive at: {FINAL}")

✅ Model weights saved securely in Google Drive at: /content/drive/MyDrive/java_bugfix_project/codet5_v1/final_production_model


# Cell 10: Real Evaluation Pipeline

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("🧹 VRAM cleared of heavy training artifacts!")

🧹 VRAM cleared of heavy training artifacts!


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

def evaluate_split(pairs: list, split_name: str, n: int = 500) -> dict:
    pairs = pairs[:n]
    BATCH = 32
    preds, targets = [], []

    for i in range(0, len(pairs), BATCH):
        batch = pairs[i:i+BATCH]
        inp = tokenizer(
            [PREFIX + p["buggy_code"] for p in batch],
            return_tensors="pt",
            max_length=MAX_IN,
            truncation=True,
            padding=True,
        ).to(device)

        with torch.no_grad():
            out = model.generate(
                **inp,
                max_new_tokens=MAX_OUT,
                num_beams=4,
            )

        preds.extend([tokenizer.decode(o, skip_special_tokens=True).strip() for o in out])
        targets.extend([[p["fixed_code"].strip()] for p in batch])

    bleu  = bleu_metric.compute(predictions=preds, references=targets)
    exact = sum(p == t[0] for p, t in zip(preds, targets))
    sim   = sum(SequenceMatcher(None, p, t[0]).ratio() for p, t in zip(preds, targets)) / len(preds)

    return {
        "split": split_name,
        "bleu": round(bleu["score"], 2),
        "exact_match_pct": round(exact / len(preds) * 100, 2),
        "similarity_pct": round(sim * 100, 1),
    }

print("Running deep validation metrics against newly trained CodeT5 model...")
val_results = evaluate_split(val_raw, "val")
print(f"📊 Results -> BLEU: {val_results['bleu']} | Exact Match: {val_results['exact_match_pct']}%")

Running deep validation metrics against newly trained CodeT5 model...
📊 Results -> BLEU: 30.68 | Exact Match: 0.4%


# Cell 11 - Test

In [ ]:
import shutil
from pathlib import Path

# Path where your trainer saved its incremental checkpoints
checkpoint_dir = Path("/content/drive/MyDrive/java_bugfix_project/codet5_v1/")

print("🔍 Searching for redundant intermediate checkpoints...")
for folder in checkpoint_dir.glob("checkpoint-*"):
    if folder.is_dir():
        print(f"🗑️ Deleting redundant snapshot: {folder.name}...")
        shutil.rmtree(folder)

print("\n✨ Clean up complete! Only your absolute final production weights remain.")

🔍 Searching for redundant intermediate checkpoints...

✨ Clean up complete! Only your absolute final production weights remain.


In [ ]:
import torch, gc
# Delete explicit references if they exist in the workspace
if 'model' in locals():
    del model
gc.collect()
torch.cuda.empty_cache()
print("🧹 GPU Memory completely cleared of previous evaluation runs!")

🧹 GPU Memory completely cleared of previous evaluation runs!


In [ ]:
# Cell 11: Final Test Set Evaluation & Qualitative Visual Inspection
import torch
import random
from transformers import AutoModelForSeq2SeqLM
from difflib import SequenceMatcher

# 1. Reload the clean production model into our newly cleared GPU space
MODEL_PATH = "/content/drive/MyDrive/java_bugfix_project/codet5_v1/final_production_model"
print("Loading production model checkpoint from Drive...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

print("\n🚀 Step 1: Running final evaluation against the UNSEEN Test Dataset...")
test_results = evaluate_split(test_raw, "test")

print("\n==================================================")
print("             🏆 FINAL TEST DATASET METRICS         ")
print("==================================================")
print(f"📊 BLEU Score:       {test_results['bleu']}")
print(f"🎯 Exact Match:      {test_results['exact_match_pct']}%")
print(f"📝 Edit Similarity:  {test_results['similarity_pct']}%")
print("==================================================\n")

print("🔍 Step 2: Extracting sample corrections for a code audit...")

# Inspect 3 random samples from the test set to look at the actual code
random.seed(42)
sample_indices = random.sample(range(len(test_raw[:500])), 3)

for idx in sample_indices:
    sample = test_raw[idx]
    buggy_code = sample["buggy_code"].strip()
    ground_truth = sample["fixed_code"].strip()

    # Generate the prediction dynamically using our safe inference framework
    inp = tokenizer(
        PREFIX + buggy_code,
        return_tensors="pt",
        max_length=MAX_IN,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=MAX_OUT, num_beams=4)

    predicted_fix = tokenizer.decode(out[0], skip_special_tokens=True).strip()

    print("-" * 60)
    print(f"🛠️ [TEST RECORD #{idx}]")
    print("-" * 60)
    print(f"❌ BUGGY INPUT:\n{buggy_code}\n")
    print(f"🤖 MODEL'S PROPOSED FIX:\n{predicted_fix}\n")
    print(f"✅ HUMAN DEVELOPER FIX:\n{ground_truth}\n")

print("-" * 60)
print("🏁 Project audit complete!")

Loading production model checkpoint from Drive...

🚀 Step 1: Running final evaluation against the UNSEEN Test Dataset...

             🏆 FINAL TEST DATASET METRICS         
📊 BLEU Score:       21.53
🎯 Exact Match:      0.0%
📝 Edit Similarity:  61.1%

🔍 Step 2: Extracting sample corrections for a code audit...
------------------------------------------------------------
🛠️ [TEST RECORD #327]
------------------------------------------------------------
❌ BUGGY INPUT:
@Override
    public ByteBuffer nioBuffer(int index, int length) {
        checkIndex(index, length);
        return ((ByteBuffer) buffer.duplicate().position(index).limit(index + length)).slice();
    }

    @Override

🤖 MODEL'S PROPOSED FIX:
@Override
    public ByteBuffer nioBuffer(int index, int length) {
        checkIndex(index, length);
        return ((ByteBuffer) buffer.duplicate().position(index).limit(index + length)).slice();
    }

    @Override

✅ HUMAN DEVELOPER FIX:
@Override
    public ByteBuffer nioBuffer(i